# CO2Dot multi-device calibration

Detect **every** CO2Dot plugged into this PC over USB and read a `spec_flash`
(dark / lit / difference spectra) from each one.

`CO2Dot.connect()` stops at the *first* device it finds, so for a multi-device
bench we scan every serial port ourselves and keep every connection that answers
`hello` as a CO2Dot. Each port is opened **once and kept open** — opening a
USB-CDC port resets the MCU, so we never open/close/re-open.

Prereqs: run from this folder so `import co2dot` resolves, and all CO2Dots plugged in.

In [1]:
from co2dot import CO2Dot, list_ports


def connect_all(hello_timeout_s=4.0):
    """Open every CO2Dot on the PC and return a list of connected handles.

    Scans all serial ports; for each, uses CO2Dot.connect(port=...), which opens
    the port once, waits out the USB-CDC reset + firmware boot, and keeps the
    connection if the device answers `hello` as a CO2Dot. Ports that are not a
    CO2Dot (or are busy) are skipped. Connections stay open — close them with
    the cleanup cell at the bottom when done.
    """
    dots = []
    for port in list_ports():
        try:
            dot = CO2Dot.connect(port=port, hello_timeout_s=hello_timeout_s)
        except Exception as e:
            print(f"  {port}: skipped ({e})")
            continue
        print(f"  found {dot.device} on {dot.port}")
        dots.append(dot)
    return dots


print("scanning serial ports ...")
dots = connect_all()
print(f"\n{len(dots)} CO2Dot(s) connected: {[d.port for d in dots]}")

scanning serial ports ...
  found CO2Dot on COM8
  COM16: skipped (No CO2Dot found. Last: COM16: could not open port 'COM16': PermissionError(13, 'Toegang geweigerd.', None, 5))
  found CO2Dot on COM19
  found CO2Dot on COM20

3 CO2Dot(s) connected: ['COM8', 'COM19', 'COM20']


In [ ]:
import json
from datetime import datetime
from pathlib import Path

# Send `spec_flash,1` to every connected CO2Dot and read back its dark/lit/diff
# spectra. Devices are handled independently so one failure does not stop the rest.
LED_CURRENT = 1

if not dots:
    raise RuntimeError("No CO2Dots connected — run the scan cell above first.")

results = {}
for dot in dots:
    try:
        flash = dot.spec_flash(LED_CURRENT)     # sends `spec_flash,1`; reads dark / lit / diff
        results[dot.port] = flash
        print(f"{dot.port} [{dot.device}] {flash['model']}: "
              f"{len(flash['diff'])} channels  diff={flash['diff']}")
    except Exception as e:
        print(f"{dot.port}: spec_flash FAILED ({e})")

print(f"\nread spec_flash from {len(results)}/{len(dots)} device(s)")

# Save all devices' spectra to one timestamped JSON (keyed by serial port).
outdir = Path("co2dot_data")
outdir.mkdir(parents=True, exist_ok=True)
record = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "led_current": LED_CURRENT,
    "devices": results,
}
path = outdir / (datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + "_multidevice_specflash.json")
path.write_text(json.dumps(record, indent=2), encoding="utf-8")
print("saved", path)

COM8 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 289, 'f2_445': 772, 'f3_480': 526, 'f4_515': 1306, 'f5_555': 2640, 'f6_590': 2425, 'f7_630': 2449, 'f8_680': 4023, 'clear': 3437, 'nir': 330}
COM19 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 288, 'f2_445': 785, 'f3_480': 471, 'f4_515': 1171, 'f5_555': 2366, 'f6_590': 2198, 'f7_630': 2300, 'f8_680': 3765, 'clear': 2917, 'nir': 279}
COM20 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 276, 'f2_445': 683, 'f3_480': 497, 'f4_515': 1210, 'f5_555': 2376, 'f6_590': 2269, 'f7_630': 2316, 'f8_680': 3764, 'clear': 2943, 'nir': 295}

read spec_flash from 3/3 device(s)
saved co2dot_data\2026-07-14_16-51-49_multidevice_specflash.json


In [ ]:
# Close every open connection when finished (frees the serial ports).
for dot in dots:
    dot.close()
print(f"closed {len(dots)} connection(s)")

closed 3 connection(s)
